# 06. 시스템 조사와 안전한 스크립팅


## Goal

로컬 시스템 정보를 읽기 전용으로 수집하고 실패에 안전한 스크립트 골격을 사용합니다.


## Setup


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-06-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"새 임시 실습 디렉터리: {lab_dir}")


## Steps

### 1. 읽기 전용 상태 스냅샷


In [ ]:
%%bash
set -euo pipefail
report="$BASH_LAB_DIR/system-snapshot.txt"
{
  printf '== timestamp ==\n'
  TZ=Asia/Seoul date '+%Y-%m-%dT%H:%M:%S%z'
  printf '\n== kernel ==\n'
  uname -a
  printf '\n== filesystem ==\n'
  df -h .
  printf '\n== current process ==\n'
  ps -p "$$" -o pid,ppid,comm,args
} | tee "$report"
test -s "$report"


### 2. 입력 경로를 신뢰하기 전에 검증


In [ ]:
%%bash
set -euo pipefail
safe_read() {
  local requested=$1
  case $requested in
    "$BASH_LAB_DIR"/*) ;;
    *) printf '범위 밖 경로 거부: %s\n' "$requested" >&2; return 64 ;;
  esac
  [[ -f $requested && -r $requested ]] || return 66
  cat -- "$requested"
}

safe_read "$BASH_LAB_DIR/system-snapshot.txt" | sed -n '1,4p'
if safe_read /etc/hosts >/dev/null 2>&1; then
  printf '예상하지 못한 허용\n'
else
  printf '범위 밖 경로를 정상적으로 거부했습니다.\n'
fi


### 3. 종료 처리 등록


In [ ]:
%%bash
set -euo pipefail
demo="$BASH_LAB_DIR/trap-demo.sh"
cat > "$demo" <<'BASH'
#!/usr/bin/env bash
set -euo pipefail
tmp=$(mktemp -d)
cleanup() { rm -rf -- "$tmp"; }
trap cleanup EXIT
printf 'temporary=%s\n' "$tmp"
printf 'done\n' > "$tmp/result.txt"
BASH
bash "$demo"


## Checks

- 스냅샷에 시각, 커널, 파일시스템, 프로세스 정보가 포함되는가?
- 허용된 임시 디렉터리 밖의 경로를 거부하는가?
- `trap ... EXIT`가 정상 종료와 오류 종료 모두에서 정리를 보장하는 이유를 설명할 수 있는가?


## Next Steps

스크립트를 반복 실행 가능한 도구로 만들고 자동 점검을 추가합니다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
